# Pruned Run Analysis (Per-Model Heatmaps + Tables)

This notebook analyzes pruning outputs from `src/run_pruned.py` under `data/benchmarks/pruning/<model>/<setting>/`.

It keeps the loading + evaluation flow and then produces, for each model:
- a matrix with pruning settings as rows and datasets as columns
- a heatmap and matching table for **correct percentage** on the base-verdict-true subset
- a heatmap and matching table for **number of correct** on the base-verdict-true subset

Settings used here: `top_1`, `top_2`, `bottom_pct_1`, `bottom_pct_2`, `bottom_pct_5`, `bottom_pct_10` for all listed models.
`top_5` and `top_10` are included only for the 6 models enabled in `scripts/run/run_pruned_sweep.sh`.

`NA` means that pruning output is not available for that model/setting/dataset.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src").exists():
    ROOT = ROOT.parent

if not (ROOT / "src").exists():
    raise RuntimeError("Could not locate repository root containing src/.")

ROOT

In [ ]:
MODEL_DIRS = [
    "deepseek-ai--DeepSeek-R1-Distill-Qwen-7B",
    "deepseek-ai--DeepSeek-R1-Distill-Llama-8B",
    "deepseek-ai--DeepSeek-R1-0528-Qwen3-8B",
    "deepseek-ai--DeepSeek-R1-Distill-Qwen-1.5B",
    "deepseek-ai--DeepSeek-R1-Distill-Qwen-14B",
    "meta-llama--Llama-3.2-3B-Instruct",
    "Qwen--Qwen3-0.6B",
    "Qwen--Qwen3-4B",
    "Qwen--Qwen3-8B",
    "Qwen--Qwen3-14B",
    "microsoft--Phi-4-mini-reasoning",
]

DATASET_FILES = {
    "humaneval": "humaneval.json",
    "mbpp": "mbpp.json",
    "livecodebench": "livecodebench_v6.json",
}

BASE_SETTING_ORDER = [
    "top_1",
    "top_2",
    "bottom_pct_1",
    "bottom_pct_2",
    "bottom_pct_5",
    "bottom_pct_10",
]

TOP_5_10_MODEL_DIRS = {
    "deepseek-ai--DeepSeek-R1-Distill-Llama-8B",
    "deepseek-ai--DeepSeek-R1-0528-Qwen3-8B",
    "meta-llama--Llama-3.2-3B-Instruct",
    "Qwen--Qwen3-4B",
    "Qwen--Qwen3-8B",
    "Qwen--Qwen3-14B",
}

FULL_SETTING_ORDER = [
    "top_1",
    "top_2",
    "top_5",
    "top_10",
    "bottom_pct_1",
    "bottom_pct_2",
    "bottom_pct_5",
    "bottom_pct_10",
]

def settings_for_model(model_dir: str) -> list[str]:
    settings = ["top_1", "top_2"]
    if model_dir in TOP_5_10_MODEL_DIRS:
        settings.extend(["top_5", "top_10"])
    settings.extend(["bottom_pct_1", "bottom_pct_2", "bottom_pct_5", "bottom_pct_10"])
    return settings

BASE_BENCH = ROOT / "data" / "benchmarks"
PRUNED_ROOT = ROOT / "data" / "benchmarks" / "pruning"

BASE_BENCH, PRUNED_ROOT

In [ ]:
def load_json(path: Path):
    if not path.exists():
        return None
    with open(path) as f:
        return json.load(f)


def truthy_verdict(value) -> bool:
    if isinstance(value, bool):
        return value
    if isinstance(value, (int, float)):
        return int(value) == 1
    if isinstance(value, str):
        return value.strip().lower() in {"1", "true", "t", "yes", "y", "pass", "passed"}
    return False


def get_eval_status_for_file(path: Path):
    rows = load_json(path)
    if rows is None or not isinstance(rows, list):
        return {
            "exists": False,
            "with_solution": 0,
            "missing_verdict": 0,
        }

    with_solution = 0
    missing_verdict = 0
    for row in rows:
        if not isinstance(row, dict):
            continue
        if row.get("solution"):
            with_solution += 1
            if "verdict" not in row:
                missing_verdict += 1

    return {
        "exists": True,
        "with_solution": with_solution,
        "missing_verdict": missing_verdict,
    }


def run_evaluate_for_folder_if_needed(folder: Path, dataset_files: dict[str, str]):
    pending_by_dataset = {}
    total_pending = 0

    for dataset, filename in dataset_files.items():
        status = get_eval_status_for_file(folder / filename)
        if status["exists"] and status["missing_verdict"] > 0:
            pending_by_dataset[dataset] = status["missing_verdict"]
            total_pending += status["missing_verdict"]

    if total_pending == 0:
        return False, pending_by_dataset

    cmd = [sys.executable, str(ROOT / "src" / "evaluate.py"), str(folder)]
    print(f"  [eval] running evaluate.py for {folder} (pending verdicts={total_pending})")
    completed = subprocess.run(cmd, cwd=ROOT, check=False)
    if completed.returncode != 0:
        raise RuntimeError(
            f"evaluate.py failed for {folder} with return code {completed.returncode}"
        )

    return True, pending_by_dataset


def summarize_pruned_vs_base(base_rows, pruned_rows):
    base_true_idx = [
        i for i, row in enumerate(base_rows)
        if isinstance(row, dict) and truthy_verdict(row.get("verdict"))
    ]

    base_true = len(base_true_idx)
    if pruned_rows is None:
        return {
            "base_true": base_true,
            "attempted": np.nan,
            "solved": np.nan,
            "verdict_present": np.nan,
            "repetition_loops": np.nan,
            "errors": np.nan,
        }

    attempted = 0
    solved = 0
    verdict_present = 0
    repetition_loops = 0
    errors = 0

    for idx in base_true_idx:
        if idx >= len(pruned_rows):
            continue
        row = pruned_rows[idx] if isinstance(pruned_rows[idx], dict) else {}

        has_solution = bool((row.get("solution") or "").strip())
        if has_solution:
            attempted += 1

        if "verdict" in row:
            verdict_present += 1
        if truthy_verdict(row.get("verdict")):
            solved += 1

        if row.get("finish_reason") == "repetition_loop":
            repetition_loops += 1
        if row.get("inference_error"):
            errors += 1

    return {
        "base_true": base_true,
        "attempted": attempted,
        "solved": solved,
        "verdict_present": verdict_present,
        "repetition_loops": repetition_loops,
        "errors": errors,
    }

In [ ]:
records = []

for model_dir in MODEL_DIRS:
    model_short = model_dir.split("--", 1)[-1]
    base_dir = BASE_BENCH / model_dir
    model_settings = settings_for_model(model_dir)

    print(f"\n=== {model_short} ===")

    evaluated_now = []
    for setting in model_settings:
        setting_dir = PRUNED_ROOT / model_short / setting
        ran_eval, _ = run_evaluate_for_folder_if_needed(setting_dir, DATASET_FILES)
        if ran_eval:
            evaluated_now.append(setting)

    if evaluated_now:
        print(f"  [eval] evaluated settings now: {', '.join(evaluated_now)}")
    else:
        print("  [eval] no pending pruning evaluations detected")

    for dataset, filename in DATASET_FILES.items():
        base_path = base_dir / filename
        base_rows = load_json(base_path)
        if base_rows is None or not isinstance(base_rows, list):
            continue

        for setting in model_settings:
            pruned_path = PRUNED_ROOT / model_short / setting / filename
            pruned_rows = load_json(pruned_path)
            stats = summarize_pruned_vs_base(base_rows, pruned_rows)

            row = {
                "model_dir": model_dir,
                "model_short": model_short,
                "dataset": dataset,
                "setting": setting,
                "pruned_file_exists": pruned_rows is not None,
            }
            row.update(stats)
            records.append(row)

df = pd.DataFrame(records)

if df.empty:
    raise RuntimeError("No rows found. Verify base/pruned output paths.")

for c in ["base_true", "attempted", "solved", "verdict_present", "repetition_loops", "errors"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df["correct_count"] = df["solved"]
df["correct_pct"] = np.where(
    df["base_true"] > 0,
    100.0 * df["solved"] / df["base_true"],
    np.nan,
)

df["dataset"] = pd.Categorical(
    df["dataset"],
    categories=list(DATASET_FILES.keys()),
    ordered=True,
 )

# Don't force a global setting order - let each model have its own ordering
# The reindex in plotting will handle the correct order per model

print(f"Prepared {len(df)} rows for per-model plotting.")

In [ ]:
MODEL_PLOT_ORDER = [(m, m.split("--", 1)[-1]) for m in MODEL_DIRS]
DATASET_ORDER = list(DATASET_FILES.keys())

def model_metric_matrix(frame: pd.DataFrame, model_short: str, value_col: str, setting_order: list[str]) -> pd.DataFrame:
    subset = frame[frame["model_short"] == model_short]
    matrix = subset.pivot_table(
        index="setting",
        columns="dataset",
        values=value_col,
        aggfunc="first",
    )
    return matrix.reindex(index=setting_order, columns=DATASET_ORDER)

def model_count_base_matrix(frame: pd.DataFrame, model_short: str, setting_order: list[str]) -> tuple:
    subset = frame[frame["model_short"] == model_short]
    count_matrix = subset.pivot_table(
        index="setting",
        columns="dataset",
        values="correct_count",
        aggfunc="first",
    )
    base_matrix = subset.pivot_table(
        index="setting",
        columns="dataset",
        values="base_true",
        aggfunc="first",
    )
    count_matrix = count_matrix.reindex(index=setting_order, columns=DATASET_ORDER)
    base_matrix = base_matrix.reindex(index=setting_order, columns=DATASET_ORDER)
    return count_matrix, base_matrix

def display_matrix_table_with_counts(pct_matrix: pd.DataFrame, count_matrix: pd.DataFrame, base_matrix: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for idx in pct_matrix.index:
        row_data = []
        for col in pct_matrix.columns:
            pct_val = pct_matrix.loc[idx, col]
            cnt_val = count_matrix.loc[idx, col]
            base_val = base_matrix.loc[idx, col]
            if pd.isna(pct_val):
                row_data.append("NA")
            else:
                row_data.append(f"{pct_val:.1f}% ({int(cnt_val)}/{int(base_val)})")
        rows.append(row_data)
    return pd.DataFrame(rows, index=pct_matrix.index, columns=pct_matrix.columns)

def plot_pct_matrix_with_counts(pct_matrix: pd.DataFrame, count_matrix: pd.DataFrame, base_matrix: pd.DataFrame, title: str):
    values = pct_matrix.astype(float)
    mask = values.isna()
    masked_values = np.ma.masked_where(mask.to_numpy(), values.to_numpy())

    cmap = plt.get_cmap("YlGnBu").copy()
    cmap.set_bad("#e8e8e8")

    _, ax = plt.subplots(figsize=(12, 5))
    sns.heatmap(
        masked_values,
        cmap=cmap,
        vmin=0,
        vmax=100,
        cbar_kws={"label": "Correct (%)"},
        linewidths=0.5,
        linecolor="white",
        xticklabels=values.columns,
        yticklabels=values.index,
        annot=False,
        ax=ax,
    )

    for i in range(values.shape[0]):
        for j in range(values.shape[1]):
            if mask.iat[i, j]:
                text = "NA"
            else:
                pct = values.iat[i, j]
                cnt = int(round(count_matrix.iat[i, j]))  
                base = int(round(base_matrix.iat[i, j]))
                text = f"{pct:.1f}%\n({cnt}/{base})"
            ax.text(j + 0.5, i + 0.5, text, ha="center", va="center", color="black", fontsize=9)

    ax.set_title(title, fontsize=12)
    ax.set_xlabel("Dataset", fontsize=11)
    ax.set_ylabel("Pruning setting", fontsize=11)
    plt.tight_layout()
    plt.show()

for model_dir, model_short in MODEL_PLOT_ORDER:
    setting_order = settings_for_model(model_dir)
    pct_matrix = model_metric_matrix(df, model_short, "correct_pct", setting_order)
    count_matrix, base_matrix = model_count_base_matrix(df, model_short, setting_order)
    print(f"\n{model_short} | Correct % on Base-Verdict-True Subset")
    display(display_matrix_table_with_counts(pct_matrix, count_matrix, base_matrix))
    plot_pct_matrix_with_counts(
        pct_matrix,
        count_matrix,
        base_matrix,
        title=f"{model_short} | Correct % on Base-Verdict-True Subset",
    )